## 检查指定交易所和交易对的 Orderbook 数据数据缺失情况

根据您的报错日志（`[WARNING] OrderbookCollector: [Orderbook][bitstamp] BTC/USD Data timeout...`），此脚本专门用于检查 **Bitstamp** 交易所 **BTC/USD** 交易对的 Orderbook 是否生成了空文件（仅有表头无实际数据）。

In [ ]:
import os
import pandas as pd
from pathlib import Path

# ================= 配置参数 =================
# 根据您的日志截图默认设置
EXCHANGE = 'bitstamp'
SYMBOL = 'BTC_USD'  # 在本地文件夹中，斜杠 '/' 通常会被替换为 '_'
DATE = '2026-03-08' 
DATA_TYPE = 'orderbooks'
# ==========================================

data_root = Path(os.path.abspath('../data/raw'))
target_dir = data_root / DATA_TYPE

print(f"🔍 开始检查 {EXCHANGE} 交易所 {SYMBOL} 在 {DATE} 的 {DATA_TYPE} 数据...\n")

found_target = False

if target_dir.exists():
    # 遍历市场类型（spot, swap 等），寻找我们需要的组合
    for market_dir in target_dir.glob('market_type=*'):
        exchange_dir = market_dir / f'exchange={EXCHANGE}'
        symbol_dir = exchange_dir / f'symbol={SYMBOL}'
        date_dir = symbol_dir / f'date={DATE}'
        
        if date_dir.exists():
            found_target = True
            market_type = market_dir.name.split('=')[1]
            print(f"📁 找到数据目录 ({market_type}): {date_dir.relative_to(data_root)}")
            
            files = list(date_dir.glob('*.parquet'))
            print(f"✅ 共发现 {len(files)} 个 Parquet 文件。\n")
            
            empty_files = []
            normal_files = []
            corrupted_files = []
            
            for f in sorted(files):
                if 'corrupted' in f.name:
                    corrupted_files.append(f.name)
                    continue
                try:
                    df = pd.read_parquet(f)
                    if df.empty:
                        empty_files.append(f.name)
                    else:
                        normal_files.append((f.name, len(df)))
                except Exception as e:
                    corrupted_files.append(f"{f.name} (错误: {e})")
            
            # 打印统计结果
            print(f"🟢 正常有数据的文件数: {len(normal_files)}")
            if normal_files:
                print(f"   示例: [{normal_files[0][0]}] 包含 {normal_files[0][1]} 行数据")
            
            print(f"\n🟡 疑似空文件数 (仅有表头无数据): {len(empty_files)}")
            for ef in empty_files[:5]:
                print(f"   - {ef}")
            if len(empty_files) > 5:
                print(f"   ... 以及其他 {len(empty_files) - 5} 个空文件")
                
            if corrupted_files:
                print(f"\n🔴 损坏的文件数 (无法读取): {len(corrupted_files)}")
                for cf in corrupted_files[:5]:
                    print(f"   - {cf}")

if not found_target:
    print(f"❌ 未找到对应的目录，请检查 {data_root.name} 目录下是否存在该交易所/交易对的数据。")
    print(f"预期搜索的路径格式: {DATA_TYPE}/market_type=*/exchange={EXCHANGE}/symbol={SYMBOL}/date={DATE}")
